# A1 — Distributions, tails, and why means lie

**From:** "what is a histogram"  **To:** reading real latency data and explaining why dashboards report p90.

A **distribution** is nothing scary: it is *all the values something takes, and how often it takes them*. Roll a die 1000 times — the list of outcomes is a distribution. Measure 1000 response delays — also a distribution. Statistics is mostly the art of describing these piles of numbers honestly.

Two words used constantly:
- **sample** — the values you actually collected (your 1000 rolls).
- **population** — the (usually unreachable) full truth the sample stands in for (every roll this die could ever make).

Remember the loop: **predict out loud → run → say what you see.**

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

rolls = rng.integers(1, 7, size=1000)
values, counts = np.unique(rolls, return_counts=True)
for v, c in zip(values, counts):
    print(f"face {v}: {c} times")

**PREDICT first:** should those counts be exactly equal? (They were not.) Each face has probability 1/6 ≈ 167 of 1000 — but randomness wobbles around that. The wobble is *sampling noise*, and it never fully goes away; it only shrinks as the sample grows. Hold that thought for book A2.

## Your first plot, and how to read any plot
The tool below is a **histogram**: it chops the number line into buckets (bins) and draws one bar per bucket; bar height = how many values landed in it. In matplotlib:
- `fig, ax = plt.subplots()` makes a canvas (`fig`) holding one drawing area (`ax`)
- `ax.hist(data, bins=...)` draws the histogram
- always label: `ax.set_xlabel(...)` (what the values are), `ax.set_ylabel(...)` (the count), `ax.set_title(...)`

A plot you cannot read aloud — "x is …, y is …, each bar means …" — is a plot you do not understand yet.

**PREDICT:** the histogram of single rolls is flat-ish. What shape is the histogram of the SUM of two dice? (Think: how many ways to make 2 vs 7.)

In [ ]:
two_dice = rng.integers(1, 7, 1000) + rng.integers(1, 7, 1000)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(rolls, bins=np.arange(0.5, 7.5), edgecolor="white")
axes[0].set_xlabel("die face"); axes[0].set_ylabel("count"); axes[0].set_title("one die: flat (uniform)")
axes[1].hist(two_dice, bins=np.arange(1.5, 13.5), edgecolor="white")
axes[1].set_xlabel("sum of two dice"); axes[1].set_ylabel("count"); axes[1].set_title("two dice: a peak at 7")
plt.tight_layout(); plt.show()

Say what you see: one die is flat; the sum has a peak (7 has six ways to happen, 2 has one). Shapes carry meaning — and the next distinction is the one this whole book exists for.

## Symmetric vs skewed, and the three "centers"
- **mean** — the balance point: add everything, divide by n.
- **median** — line everyone up, take the middle person.
- **mode** — the most common value.

On a **symmetric** pile they coincide. On a **skewed** pile — one with a long tail of rare huge values — the mean gets dragged toward the tail while the median stays with the crowd. Latency data is almost always right-skewed: most responses are quick, a few are terrible.

**PREDICT:** in the skewed plot below, which line sits further right — mean or median?

In [ ]:
symmetric = rng.normal(500, 80, 5000)
skewed = rng.lognormal(mean=6.0, sigma=0.6, size=5000)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, data, name in [(axes[0], symmetric, "symmetric"), (axes[1], skewed, "right-skewed (long tail)")]:
    ax.hist(data, bins=50)
    ax.axvline(data.mean(), color="tab:red", lw=2, label=f"mean {data.mean():.0f}")
    ax.axvline(np.median(data), color="tab:green", lw=2, label=f"median {np.median(data):.0f}")
    ax.set_title(name); ax.set_xlabel("value"); ax.set_ylabel("count"); ax.legend()
plt.tight_layout(); plt.show()

The mean chased the tail; the median did not. Whenever someone quotes you a *mean* for skewed data (latency, income, file sizes), they are — knowingly or not — letting the tail speak for the crowd.

## Percentiles: naming positions in the pile
Sort all values. The **p-th percentile** is the value below which p% of the data sits. p50 *is* the median. **p90** answers: "how bad is the experience for the unluckiest 10%?" Production systems live and die by p90/p99 because users do not experience averages — each user experiences one draw, and the angry ones come from the tail.

Now real data: every user→agent response gap across our 11 normalized calls (the caller finishes, how long until the agent speaks — in milliseconds, from the conversation logs you helped produce).

**PREDICT:** will the mean sit above or below the median here? Where will p90 land relative to both?

In [ ]:
import json
from signals import turn_metrics
gaps = []
for p in sorted((ROOT / "data" / "normalized").glob("*.json")):
    call = json.loads(p.read_text())
    for e in turn_metrics(call["turns"]):
        if e["prev_spk"] == "user" and e["next_spk"] == "agent" and e["fto_ms"] >= 0:
            gaps.append(e["gap_ms"])
gaps = np.array(gaps)
mean, med, p90 = gaps.mean(), np.median(gaps), np.percentile(gaps, 90)
print(f"n={len(gaps)} response gaps · mean={mean:.0f}ms · median={med:.0f}ms · p90={p90:.0f}ms")
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(gaps, bins=40)
for v, lbl, c in [(mean, "mean", "tab:red"), (med, "median", "tab:green"), (p90, "p90", "tab:purple")]:
    ax.axvline(v, color=c, lw=2, label=f"{lbl} {v:.0f}ms")
ax.set_xlabel("user->agent gap (ms)"); ax.set_ylabel("count")
ax.set_title("real response gaps, 11 calls"); ax.legend(); plt.show()

Read it aloud: where is the bulk? where is the tail? which single number would you tell a founder describes "typical" — and what does p90 say that the mean hides? (Our rubric calls a gap laggy above 800ms — find that point on the x-axis and estimate how much of the pile sits beyond it.)

## Exercise — p90 with your bare hands
No `np.percentile`. Sort, find the index, look it up. Then check yourself.

In [ ]:
sorted_gaps = np.sort(gaps)
idx = int(np.ceil(0.9 * len(sorted_gaps))) - 1     # the value with 90% of the pile at or below it
by_hand = sorted_gaps[idx]
print(f"by hand: {by_hand}ms · numpy: {np.percentile(gaps, 90):.0f}ms")
print("(small differences are fine - there are several interpolation conventions; the IDEA is identical)")

## Self-check (out loud, then expand)
1. Define distribution, sample, population in one sentence each.
2. Why does the mean exceed the median on right-skewed data?
3. What question does p90 answer that the mean cannot?
4. Our pool's median gap was healthy but p90 was far beyond 800ms. Describe the user experience in plain words.
5. **Gotcha:** a release improves mean latency 20% but p90 doubles. What probably happened, and who noticed?

<details><summary>Answers</summary>

1. Distribution: the values something takes and how often. Sample: the values you collected. Population: the full truth the sample approximates.
2. The long tail of rare large values drags the sum (hence the mean); the middle-ranked value barely moves.
3. "How bad is it for the worst 10% of experiences?" — the tail's size and location.
4. Most replies feel fine; roughly one in ten exchanges has the caller hanging in silence past the point where it feels broken. Callers remember those.
5. Typical requests got faster, but some path (cache miss, retry, lock) got much slower; the unlucky minority noticed — loudly. Means hide tail regressions; that is why we report median + p90.
</details>